In [0]:
display(dbutils.fs.ls('/databricks-datasets/cs110x/ml-20m/data-001/'))

In [0]:
display(dbutils.fs.ls('/databricks-datasets/cs110x/ml-20m/data-001/'))

In [0]:
%fs head /databricks-datasets/cs110x/ml-20m/data-001/ratings.csv

In [0]:
%fs head /databricks-datasets/cs110x/ml-20m/data-001/movies.csv

In [0]:
from pyspark.sql.types import *

movies_schema = StructType([
  StructField('movieId', IntegerType()),
  StructField('title', StringType()),
  StructField('genres', StringType())
])
ratings_schema = StructType([
  StructField('userId', IntegerType()),
  StructField('movieId', IntegerType()),
  StructField('rating', FloatType()), 

])

In [0]:
file_location = "/databricks-datasets/cs110x/ml-20m/data-001/movies.csv"
file_type = "csv"

# CSV options
infer_schema = "true"
first_row_is_header = "false"


# The applied options are for CSV files. For other file types, these will be ignored.
df_movies = spark.read.format(file_type) \
  .option("inferSchema", "false") \
  .option("header", "true") \
  .schema(movies_schema) \
  .load(file_location)

display(df_movies)

In [0]:
file_location = "/databricks-datasets/cs110x/ml-20m/data-001/ratings.csv"
file_type = "csv"


infer_schema = "true"
first_row_is_header = "false"


df_ratings = spark.read.format(file_type) \
  .option("inferSchema", "false") \
  .option("header", "true") \
  .schema(ratings_schema) \
  .load(file_location)

display(df_ratings)

In [0]:
df_ratings.select('rating').describe().show()


In [0]:
df_rating_train, df_rating_test = df_ratings.randomSplit([0.8, 0.2], seed=42)
#display(df_rating_train)
print(df_rating_train.count())
print(df_rating_test.count())

In [0]:
from pyspark.ml.recommendation import ALS

als = ALS(rank=10, 
          maxIter=5,
          regParam=0.5, 
          userCol="userId",
          itemCol="movieId",
          ratingCol="rating",
          coldStartStrategy="drop")         
        

model1 = als.fit(df_rating_train)

In [0]:
from pyspark.ml.recommendation import ALS

als = ALS(rank=20, 
          maxIter=8,
          regParam=0.01, 
          userCol="userId",
          itemCol="movieId",
          ratingCol="rating",
          coldStartStrategy="drop"
          )  
                
          

model2 = als.fit(df_rating_train)

In [0]:
from pyspark.sql.functions import round, col

df_predicted_rating = model1.transform(df_rating_test)
df_predicted_rating = df_predicted_rating.filter(df_predicted_rating.prediction != float('nan'))
df_predicted_rating.select('rating', 'prediction').describe().show()
#df_predicted_rating = df_predicted_rating.withColumn('prediction rounded', round(col('prediction'), 1))

display(df_predicted_rating)


In [0]:
from pyspark.sql.functions import round, col

df_predicted_rating = model2.transform(df_rating_test)
df_predicted_rating = df_predicted_rating.filter(df_predicted_rating.prediction != float('nan'))
df_predicted_rating.select('rating', 'prediction').describe().show()
#df_predicted_rating = df_predicted_rating.withColumn('prediction rounded', round(col('prediction'), 1))

display(df_predicted_rating)

In [0]:
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(metricName="rmse", labelCol="rating", predictionCol="prediction")

def evaluate_model(model, test_df):
    predictions = model.transform(test_df)
    rmse = evaluator.evaluate(predictions)
    return rmse, predictions
rmse1, pred1 = evaluate_model(model1, df_rating_test)
rmse2, pred2 = evaluate_model(model2, df_rating_test)

print("Model 1 RMSE =", rmse1)
print("Model 2 RMSE =", rmse2)    

In [0]:
from pyspark.sql import Row
rows =[Row(Model = 'model1', rank = 10, maxIter = 5, regParam = 0.5, rmse = rmse1),
       Row(Model = 'model2', rank = 20, maxIter = 8, regParam = 0.01, rmse = rmse2)]
df_results_spark = spark.createDataFrame(rows)
display(df_results_spark)

Databricks visualization. Run in Databricks to view.

In [0]:
my_user_id = 0
my_rated_movies = [
    (my_user_id, 364, 5), 
    (my_user_id, 520, 4), 
    (my_user_id, 521, 5), 
    (my_user_id, 875, 4), 
    (my_user_id, 1036, 5), 
    (my_user_id, 734, 3), 
    (my_user_id, 1054, 4), 
    (my_user_id, 904, 4),
    (my_user_id, 760, 5),  
    (my_user_id, 1243, 5), 
]
print(my_rated_movies)

In [0]:
df_custom_ratings = spark.createDataFrame(my_rated_movies, ["userId", "movieId", "ratings"])
display(df_custom_ratings)

In [0]:
df_all_ratings = df_rating_train.union(df_custom_ratings)
print(df_all_ratings.count())
display(df_all_ratings)

In [0]:
from pyspark.ml.recommendation import ALS

als = ALS(rank=10, 
          maxIter=5,
          regParam=0.05, 
          userCol="userId",
          itemCol="movieId",
          ratingCol="rating",
          coldStartStrategy="drop")         
          #implicitPrefs=False

custom_model = als.fit(df_all_ratings)

In [0]:
from pyspark.sql.functions import lit, col, desc
 
print(f'movies before: {df_movies.count()}')
df_movies_unrated = df_movies.join(df_custom_ratings, on="movieId", how="left_anti")
print(f'movies after: {df_movies_unrated.count()}')
 
df_for_prediction = df_movies_unrated.withColumn("userId", lit(0))
df_predictions = custom_model.transform(df_for_prediction)
df_recommendations = df_predictions.filter(df_predictions.prediction != float('nan')) \
                                   .orderBy(desc("prediction")).limit(20)
display(df_recommendations.select("title", "genres", "prediction"))